In [5]:
import numpy as np
import pandas as pd
from nilearn.input_data import NiftiLabelsMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
import os

func_fn_b = '/disk1/MRI-Data_in-use/20_narrativefMRI/10_ds002245-v.1.0.3_Hasson/derivatives/fmriprep_v25.1.4/'
# -------------------------------------------------------
# Task selection
# -------------------------------------------------------
task = input("task name (piemanpni / bronx): ").strip()
list = pd.read_csv("../list/participants_piemanpni.tsv", sep="\t")

# 1列目（インデックス0）の要素を配列（リスト）として保存
sub = list.iloc[:, 0].tolist()

if task == "piemanpni":
    start_trs = 14
    end_trs = 13
    valid_trs = 267   # ← 音声区間の長さ
elif task == "bronx":
    start_trs = 18
    end_trs = 14
    valid_trs = 357   # ← 音声区間の長さ
else:
    raise ValueError(f"Unknown task: {task}")

print(f"=== Task: {task} | start_TR={start_trs}, end_TR={end_trs}, valid_TR={valid_trs} ===")

# -------------------------------------------------------
# Schaefer atlas
# -------------------------------------------------------
n_parcels = 400
atlas = fetch_atlas_schaefer_2018(n_rois=n_parcels, yeo_networks=17, resolution_mm=2)
masker = NiftiLabelsMasker(atlas.maps, labels=atlas.labels, standardize=True)

rows = []

print("=== NIfTI → 全被験者のパーセル時系列を読み込み → CSV 保存 ===")

# -------------------------------------------------------
# Loop subjects
# -------------------------------------------------------
for subname in sub:
    func_fn = (
        f"{func_fn_b}{subname}/func/"
        f"{subname}_task-{task}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    )

    try:
        ts = masker.fit_transform(func_fn)  # (TR, parcel)
        total_trs = ts.shape[0]

        expected_total = start_trs + valid_trs + end_trs

        # TR 長が想定と一致しているかチェック
        if total_trs != expected_total:
            raise ValueError(
                f"TR mismatch: found {total_trs}, expected {expected_total}"
            )

        # 切り出し（有効区間だけ残す）
        ts_cut = ts[start_trs : start_trs+valid_trs]

        # パーセルごとに保存
        for tr in range(ts_cut.shape[0]):
            row = {"sub": subname, "TR": tr}
            row.update({f"parcel_{i}": ts_cut[tr, i] for i in range(n_parcels)})
            rows.append(row)

        print(f"✓ {subname} saved")

    except Exception as e:
        print(f"⚠ {subname} skipped ({e})")

# -------------------------------------------------------
# Save CSV
# -------------------------------------------------------
out_fn = f"../parcels_csv/{task}_parcels_all_subjects.csv"
df_all = pd.DataFrame(rows)
df_all.to_csv(out_fn, index=False)

print(f"✅ Saved: {out_fn}")


task name (piemanpni / bronx):  piemanpni


=== Task: piemanpni | start_TR=14, end_TR=13, valid_TR=267 ===
[fetch_atlas_schaefer_2018] Dataset found in /home/y-sato/nilearn_data/schaefer_2018
=== NIfTI → 全被験者のパーセル時系列を読み込み → CSV 保存 ===
✓ sub-127 saved
✓ sub-265 saved
✓ sub-267 saved
✓ sub-272 saved
✓ sub-273 saved
✓ sub-274 saved
✓ sub-275 saved
✓ sub-276 saved
✓ sub-277 saved
✓ sub-278 saved
✓ sub-279 saved
⚠ sub-280 skipped (File not found: '/disk1/MRI-Data_in-use/20_narrativefMRI/10_ds002245-v.1.0.3_Hasson/derivatives/fmriprep_v25.1.4/sub-280/func/sub-280_task-piemanpni_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz')
✓ sub-281 saved
✓ sub-282 saved
✓ sub-283 saved
✓ sub-284 saved
✓ sub-285 saved
✓ sub-286 saved
✓ sub-287 saved
✓ sub-288 saved
✓ sub-289 saved
✓ sub-290 saved
✓ sub-291 saved
✓ sub-292 saved
✓ sub-293 saved
✓ sub-294 saved
✓ sub-295 saved
✓ sub-296 saved
✓ sub-297 saved
✓ sub-298 saved
✓ sub-299 saved
✓ sub-300 saved
✓ sub-301 saved
✓ sub-302 saved
✓ sub-303 saved
✓ sub-304 saved
✓ sub-305 saved
✓ sub-306 sa

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -------------------------------------------------------
# task selection
# -------------------------------------------------------
task = input("task name (piemanpni / bronx): ").strip()

# 入力 CSV（未正規化）
csv_path = f"../parcels_csv/{task}_parcels_all_subjects.csv"

df = pd.read_csv(csv_path)
print("Loaded:", csv_path)

# -------------------------------------------------------
# Extract subjects, TR, parcels
# -------------------------------------------------------

valid_subs = sorted(df["sub"].unique())
n_TR = df["TR"].max() + 1
parcel_cols = [c for c in df.columns if c.startswith("parcel_")]
n_parcels = len(parcel_cols)

print(f"subjects={len(valid_subs)}, TR={n_TR}, parcels={n_parcels}")

# -------------------------------------------------------
# Create 3D array
# all_ts[subject, TR, parcel]
# -------------------------------------------------------

all_ts = np.zeros((len(valid_subs), n_TR, n_parcels))

for i, subname in enumerate(valid_subs):
    sub_df = df[df["sub"] == subname].sort_values("TR")
    all_ts[i] = sub_df[parcel_cols].values

# -------------------------------------------------------
# Min-Max Normalization (0–1)
# 全体（subjects×TR×parcels）をまとめて正規化
# -------------------------------------------------------

subjects = len(valid_subs)
all_ts_2d = all_ts.reshape(subjects, n_TR * n_parcels)

scaler = MinMaxScaler()
all_ts_scaled_2d = scaler.fit_transform(all_ts_2d)

all_ts_scaled = all_ts_scaled_2d.reshape(subjects, n_TR, n_parcels)

# -------------------------------------------------------
# Save normalized CSV
# -------------------------------------------------------

rows = []
for si, subname in enumerate(valid_subs):
    for tr in range(n_TR):
        row = {"sub": subname, "TR": tr}
        row.update({f"parcel_{p}": all_ts_scaled[si, tr, p] for p in range(n_parcels)})
        rows.append(row)

df_out = pd.DataFrame(rows)
out_fn = f"../parcels_csv/{task}_parcels_all_subjects_normalized.csv"
df_out.to_csv(out_fn, index=False)

print(f"✅ Normalized CSV saved: {out_fn}")


task name (piemanpni / bronx):  bronx


Loaded: ../parcels_csv/bronx_parcels_all_subjects.csv
subjects=47, TR=358, parcels=400
✅ Normalized CSV saved: ../parcels_csv/bronx_parcels_all_subjects_normalized.csv
